<a href="https://colab.research.google.com/github/Suyash-Codes-AI/Hack4Delhi-Submission/blob/master/RISK_SCORING_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas scikit-learn joblib

In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import joblib


le_dept = LabelEncoder()
le_state = LabelEncoder()

def train_risk_model():
    spending_df = pd.read_csv('govt_spending_demo_upload_2.csv')
    audit_df = pd.read_csv('Delhi Audit Dataset.csv')


    def clean_money(val):
        if isinstance(val, str):
            if 'Unknown' in val: return 0
            return float(val.replace(',', '').replace('E+11', '00000000000'))
        return val

    audit_df['Financial_Implication_Clean'] = audit_df['Financial_Implication_INR'].apply(clean_money)


    audit_risk_map = audit_df.groupby('Department')['Financial_Implication_Clean'].sum().to_dict()


    def get_audit_risk(dept_name):

        for key, val in audit_risk_map.items():
            if key in dept_name or dept_name in key:
                return np.log1p(val)
        return 0

    spending_df['Audit_History_Score'] = spending_df['department'].apply(get_audit_risk)



    spending_df['dept_code'] = le_dept.fit_transform(spending_df['department'])
    spending_df['state_code'] = le_state.fit_transform(spending_df['state'])

    X = spending_df[['amount', 'beneficiaryCount', 'Audit_History_Score', 'dept_code', 'state_code']]
    y = spending_df['riskScore']


    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)


    artifacts = {
        'model': model,
        'le_dept': le_dept,
        'le_state': le_state,
        'audit_risk_map': audit_risk_map
    }
    joblib.dump(artifacts, 'ners_risk_model.pkl')
    print("Model trained and saved as 'ners_risk_model.pkl'")

def predict_risk(transaction_data):
    try:
        artifacts = joblib.load('ners_risk_model.pkl')
    except:
        return {"error": "Model not found. Run train_risk_model() first."}

    model = artifacts['model']
    le_dept = artifacts['le_dept']
    le_state = artifacts['le_state']
    audit_map = artifacts['audit_risk_map']


    dept = transaction_data['department']


    audit_score = 0
    for key, val in audit_map.items():
        if key in dept or dept in key:
            audit_score = np.log1p(val)
            break


    try:
        d_code = le_dept.transform([dept])[0]
    except:
        d_code = 0

    try:
        s_code = le_state.transform([transaction_data['state']])[0]
    except:
        s_code = 0


    feature_columns = ['amount', 'beneficiaryCount', 'Audit_History_Score', 'dept_code', 'state_code']
    features_df = pd.DataFrame([[
        transaction_data['amount'],
        transaction_data['beneficiaryCount'],
        audit_score,
        d_code,
        s_code
    ]], columns=feature_columns)

    predicted_score = model.predict(features_df)[0]


    level = "Low"
    if predicted_score > 75:
        level = "High"
    elif predicted_score > 40:
        level = "Medium"

    return {
        "risk_score": round(predicted_score, 2),
        "risk_level": level,
        "details": f"Score calculated based on amount ₹{transaction_data['amount']} and audit history."
    }


if __name__ == "__main__":

    train_risk_model()


    sample_txn = {
        'amount': 15000000,
        'beneficiaryCount': 0,
        'department': 'PWD',
        'state': 'Delhi'
    }

    result = predict_risk(sample_txn)
    print("\nPrediction Result:")
    print(result)


Model trained and saved as 'ners_risk_model.pkl'

Prediction Result:
{'risk_score': np.float64(56.56), 'risk_level': 'Medium', 'details': 'Score calculated based on amount ₹15000000 and audit history.'}
